# Model Comparison: FunctionGemma 270M vs LFM2 350M

Evaluate both fine-tuned models on:
- **Test set** (725 examples from `test.jsonl`)
- **Stress test** (93 handwritten edge cases from `handwritten.jsonl`)

Metrics: safe accuracy, severity accuracy, severity within-1, principle F1, risk label F1, valid JSON rate.

In [ ]:
!pip install -q transformers>=4.46 peft>=0.13 datasets>=3.0 accelerate>=1.0 bitsandbytes>=0.44 sacrebleu

import json
import torch
from tqdm import tqdm
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from collections import defaultdict
import shared_utils as utils

In [ ]:
TEST_PATH = "test.jsonl"
STRESS_PATH = "handwritten.jsonl"

test_data = utils.load_jsonl(TEST_PATH)
stress_data = utils.load_jsonl(STRESS_PATH)

print(f"Test set: {len(test_data)} examples")
print(f"Stress test: {len(stress_data)} examples")

In [ ]:
def run_inference(model, tokenizer, examples, prompt_fn, batch_desc="Inferring"):
    """Run inference on a list of examples. Returns (raw_outputs, parsed_outputs)."""
    model.eval()
    raw_outputs = []
    parsed_outputs = []

    for ex in tqdm(examples, desc=batch_desc):
        prompt = prompt_fn(ex["input_text"])
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=300,
                do_sample=False,
                temperature=1.0,
            )

        response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
        parsed = utils.parse_model_output(response)

        raw_outputs.append(parsed)
        parsed_outputs.append(parsed if parsed else {})

    return raw_outputs, parsed_outputs

## Load FunctionGemma 270M (fine-tuned)

In [ ]:
GEMMA_BASE = "google/functiongemma-270m-it"
GEMMA_ADAPTER = "gemma3-malaysia-mod/final_adapter"

gemma_tokenizer = AutoTokenizer.from_pretrained(GEMMA_ADAPTER)
gemma_base = AutoModelForCausalLM.from_pretrained(
    GEMMA_BASE, torch_dtype=torch.float16, device_map="auto"
)
gemma_model = PeftModel.from_pretrained(gemma_base, GEMMA_ADAPTER)
gemma_model.eval()
print("FunctionGemma loaded.")

In [ ]:
gemma_test_raw, gemma_test_parsed = run_inference(
    gemma_model, gemma_tokenizer, test_data, utils.inference_prompt_gemma, "Gemma test"
)
gemma_stress_raw, gemma_stress_parsed = run_inference(
    gemma_model, gemma_tokenizer, stress_data, utils.inference_prompt_gemma, "Gemma stress"
)

In [ ]:
del gemma_model, gemma_base
torch.cuda.empty_cache()
print("Gemma freed from GPU.")

## Load LFM2 350M (fine-tuned)

In [ ]:
LFM2_BASE = "LiquidAI/LFM2-350M"
LFM2_ADAPTER = "lfm2-malaysia-mod/final_adapter"

lfm2_tokenizer = AutoTokenizer.from_pretrained(LFM2_ADAPTER, trust_remote_code=True)
lfm2_base = AutoModelForCausalLM.from_pretrained(
    LFM2_BASE, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
)
lfm2_model = PeftModel.from_pretrained(lfm2_base, LFM2_ADAPTER)
lfm2_model.eval()
print("LFM2 loaded.")

In [ ]:
lfm2_test_raw, lfm2_test_parsed = run_inference(
    lfm2_model, lfm2_tokenizer, test_data, utils.inference_prompt_lfm2, "LFM2 test"
)
lfm2_stress_raw, lfm2_stress_parsed = run_inference(
    lfm2_model, lfm2_tokenizer, stress_data, utils.inference_prompt_lfm2, "LFM2 stress"
)

In [ ]:
del lfm2_model, lfm2_base
torch.cuda.empty_cache()

## Results: Test Set (725 examples)

In [ ]:
gemma_test_metrics = utils.compute_all_metrics(gemma_test_parsed, test_data, gemma_test_raw)
lfm2_test_metrics = utils.compute_all_metrics(lfm2_test_parsed, test_data, lfm2_test_raw)

print("=== Test Set Results ===\n")
utils.print_comparison(gemma_test_metrics, lfm2_test_metrics)

## Results: Stress Test (93 edge cases)

In [ ]:
gemma_stress_metrics = utils.compute_all_metrics(gemma_stress_parsed, stress_data, gemma_stress_raw)
lfm2_stress_metrics = utils.compute_all_metrics(lfm2_stress_parsed, stress_data, lfm2_stress_raw)

print("=== Stress Test Results ===\n")
utils.print_comparison(gemma_stress_metrics, lfm2_stress_metrics)

## Stress Test Breakdown by Edge Case Type

In [ ]:
def breakdown_by_type(parsed_preds, golds):
    """Group by edge_case_type and compute severity accuracy per group."""
    groups = defaultdict(lambda: ([], []))
    for pred, gold in zip(parsed_preds, golds):
        etype = gold.get("edge_case_type", "unknown")
        groups[etype][0].append(pred)
        groups[etype][1].append(gold)

    print(f"{'Edge Case Type':<25} {'Count':>6} {'Safe Acc':>10} {'Sev Acc':>10}")
    print("-" * 55)
    for etype in sorted(groups.keys()):
        preds, gs = groups[etype]
        sa = utils.safe_accuracy(preds, gs)
        sev = utils.severity_accuracy(preds, gs)
        print(f"{etype:<25} {len(gs):>6} {sa:>9.1%} {sev:>9.1%}")

print("=== FunctionGemma 270M ===")
breakdown_by_type(gemma_stress_parsed, stress_data)

print("\n=== LFM2 350M ===")
breakdown_by_type(lfm2_stress_parsed, stress_data)

## Error Analysis: Failed JSON Outputs

In [ ]:
def show_failures(raw_outputs, examples, model_name, max_show=5):
    """Show examples where model failed to produce valid JSON."""
    failures = [(i, ex) for i, (out, ex) in enumerate(zip(raw_outputs, examples)) if out is None]
    print(f"\n{model_name}: {len(failures)} / {len(examples)} failed to produce valid JSON")
    for i, (idx, ex) in enumerate(failures[:max_show]):
        print(f"  [{idx}] {ex['input_text'][:80]}...")

show_failures(gemma_test_raw, test_data, "FunctionGemma (test)")
show_failures(gemma_stress_raw, stress_data, "FunctionGemma (stress)")
show_failures(lfm2_test_raw, test_data, "LFM2 (test)")
show_failures(lfm2_stress_raw, stress_data, "LFM2 (stress)")

## Summary

Based on the metrics above, compare the two models across all dimensions to determine the best candidate for deployment as a Malaysia content moderation judge + rewrite model.